# 03 - 简单线性回归

**目标**: 理解线性回归原理，掌握单变量线性回归的实现与评估

**预计时间**: 2-3 小时

**前置知识**: NumPy 基础, Pandas 数据处理

---

## 1. 导入库

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

# 设置中文显示
plt.rcParams['font.sans-serif'] = ['SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

print("库导入成功！")

## 2. 线性回归原理回顾

### 数学模型

线性回归假设目标变量 $y$ 与特征 $x$ 之间存在线性关系：

$$y = wx + b + \epsilon$$

其中：
- $w$ : 权重（斜率）
- $b$ : 偏置（截距）
- $\epsilon$ : 误差项

### 目标

找到最优的 $w$ 和 $b$，使得预测值与真实值的误差最小：

$$\min_{w,b} \sum_{i=1}^{n} (y_i - (wx_i + b))^2$$

这被称为**最小二乘法**（Ordinary Least Squares, OLS）

## 3. 生成模拟数据

In [ ]:
# 设置随机种子，保证结果可复现
np.random.seed(42)

# 生成模拟数据：学习时间与考试成绩的关系
n_samples = 100

# 学习时长（小时）
X = np.random.uniform(1, 10, n_samples)

# 真实关系：成绩 = 5 * 学习时长 + 60 + 噪声
true_w = 5
true_b = 60
noise = np.random.normal(0, 5, n_samples)
y = true_w * X + true_b + noise

# 创建 DataFrame
df = pd.DataFrame({
    'study_hours': X,
    'score': y
})

print("数据预览:")
print(df.head(10))
print(f"\n数据形状: {df.shape}")
print(f"\n统计摘要:")
print(df.describe())

In [ ]:
# 可视化数据
plt.figure(figsize=(10, 6))
plt.scatter(df['study_hours'], df['score'], alpha=0.6, color='blue', label='数据点')

# 绘制真实关系线
x_line = np.linspace(1, 10, 100)
y_true = true_w * x_line + true_b
plt.plot(x_line, y_true, 'r--', linewidth=2, label=f'真实关系: y={true_w}x+{true_b}')

plt.xlabel('学习时长（小时）')
plt.ylabel('考试成绩')
plt.title('学习时长与考试成绩的关系')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 4. 手动实现线性回归（最小二乘法）

In [ ]:
class SimpleLinearRegression:
    """
    简单线性回归的 NumPy 实现
    使用正规方程（Normal Equation）求解
    """
    
    def __init__(self):
        self.w = None  # 权重
        self.b = None  # 偏置
    
    def fit(self, X, y):
        """
        训练模型
        
        参数:
            X: 特征，shape (n_samples,)
            y: 目标，shape (n_samples,)
        """
        n = len(X)
        
        # 计算均值
        x_mean = np.mean(X)
        y_mean = np.mean(y)
        
        # 计算 w（斜率）
        numerator = np.sum((X - x_mean) * (y - y_mean))
        denominator = np.sum((X - x_mean) ** 2)
        self.w = numerator / denominator
        
        # 计算 b（截距）
        self.b = y_mean - self.w * x_mean
        
        return self
    
    def predict(self, X):
        """预测"""
        return self.w * X + self.b
    
    def score(self, X, y):
        """计算 R² 分数"""
        y_pred = self.predict(X)
        ss_res = np.sum((y - y_pred) ** 2)
        ss_tot = np.sum((y - np.mean(y)) ** 2)
        return 1 - (ss_res / ss_tot)

# 测试手动实现
model_manual = SimpleLinearRegression()
model_manual.fit(df['study_hours'].values, df['score'].values)

print("手动实现结果:")
print(f"权重 w = {model_manual.w:.4f}")
print(f"偏置 b = {model_manual.b:.4f}")
print(f"真实值: w = {true_w}, b = {true_b}")
print(f"R² 分数 = {model_manual.score(df['study_hours'].values, df['score'].values):.4f}")

In [ ]:
# 可视化手动实现的结果
plt.figure(figsize=(10, 6))
plt.scatter(df['study_hours'], df['score'], alpha=0.6, color='blue', label='数据点')

# 绘制拟合线
x_line = np.linspace(1, 10, 100)
y_pred_manual = model_manual.predict(x_line)
plt.plot(x_line, y_pred_manual, 'g-', linewidth=2, 
         label=f'拟合线: y={model_manual.w:.2f}x+{model_manual.b:.2f}')

# 绘制真实线
y_true = true_w * x_line + true_b
plt.plot(x_line, y_true, 'r--', linewidth=2, alpha=0.7, label=f'真实线: y={true_w}x+{true_b}')

plt.xlabel('学习时长（小时）')
plt.ylabel('考试成绩')
plt.title('手动实现：线性回归拟合')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 5. 使用 Scikit-learn 实现

In [ ]:
# 准备数据
X = df[['study_hours']].values  # 需要是二维数组
y = df['score'].values

# 划分训练集和测试集
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"训练集大小: {len(X_train)}")
print(f"测试集大小: {len(X_test)}")

In [ ]:
# 创建并训练模型
model_sklearn = LinearRegression()
model_sklearn.fit(X_train, y_train)

# 查看模型参数
print("Scikit-learn 模型结果:")
print(f"权重 (coef_): {model_sklearn.coef_[0]:.4f}")
print(f"偏置 (intercept_): {model_sklearn.intercept_:.4f}")
print(f"真实值: w = {true_w}, b = {true_b}")

# 预测
y_train_pred = model_sklearn.predict(X_train)
y_test_pred = model_sklearn.predict(X_test)

# 评估
print(f"\n训练集 R²: {r2_score(y_train, y_train_pred):.4f}")
print(f"测试集 R²: {r2_score(y_test, y_test_pred):.4f}")
print(f"训练集 RMSE: {np.sqrt(mean_squared_error(y_train, y_train_pred)):.4f}")
print(f"测试集 RMSE: {np.sqrt(mean_squared_error(y_test, y_test_pred)):.4f}")

In [ ]:
# 可视化结果
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 训练集
axes[0].scatter(X_train, y_train, alpha=0.6, color='blue', label='训练数据')
axes[0].plot(X_train, y_train_pred, 'r-', linewidth=2, label='拟合线')
axes[0].set_xlabel('学习时长（小时）')
axes[0].set_ylabel('考试成绩')
axes[0].set_title(f'训练集 (R²={r2_score(y_train, y_train_pred):.3f})')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# 测试集
axes[1].scatter(X_test, y_test, alpha=0.6, color='green', label='测试数据')
axes[1].plot(X_test, y_test_pred, 'r-', linewidth=2, label='预测线')
axes[1].set_xlabel('学习时长（小时）')
axes[1].set_ylabel('考试成绩')
axes[1].set_title(f'测试集 (R²={r2_score(y_test, y_test_pred):.3f})')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 6. 模型评估指标详解

### 6.1 R² 决定系数

$$R^2 = 1 - \frac{SS_{res}}{SS_{tot}} = 1 - \frac{\sum(y_i - \hat{y}_i)^2}{\sum(y_i - \bar{y})^2}$$

- $R^2 = 1$ : 完美拟合
- $R^2 = 0$ : 模型等同于简单取均值
- $R^2 < 0$ : 模型比取均值还差

In [ ]:
# 手动计算 R²
def calculate_r2(y_true, y_pred):
    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)
    return 1 - (ss_res / ss_tot)

r2_manual = calculate_r2(y_test, y_test_pred)
r2_sklearn = r2_score(y_test, y_test_pred)

print(f"手动计算 R²: {r2_manual:.6f}")
print(f"sklearn R²: {r2_sklearn:.6f}")
print(f"是否相等: {np.isclose(r2_manual, r2_sklearn)}")

### 6.2 MSE 和 RMSE

**均方误差 (MSE)**: $MSE = \frac{1}{n}\sum_{i=1}^{n}(y_i - \hat{y}_i)^2$

**均方根误差 (RMSE)**: $RMSE = \sqrt{MSE}$

RMSE 与目标变量同单位，更易解释

In [ ]:
# 计算各种误差指标
mse = mean_squared_error(y_test, y_test_pred)
rmse = np.sqrt(mse)
mae = np.mean(np.abs(y_test - y_test_pred))

print("误差指标:")
print(f"MSE (均方误差): {mse:.4f}")
print(f"RMSE (均方根误差): {rmse:.4f}")
print(f"MAE (平均绝对误差): {mae:.4f}")
print(f"\n目标变量范围: [{y.min():.1f}, {y.max():.1f}]")
print(f"RMSE 占范围比例: {rmse/(y.max()-y.min())*100:.2f}%")

### 6.3 残差分析

In [ ]:
# 计算残差
residuals = y_test - y_test_pred

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# 残差 vs 预测值
axes[0].scatter(y_test_pred, residuals, alpha=0.6)
axes[0].axhline(y=0, color='r', linestyle='--')
axes[0].set_xlabel('预测值')
axes[0].set_ylabel('残差')
axes[0].set_title('残差图')
axes[0].grid(True, alpha=0.3)

# 残差分布直方图
axes[1].hist(residuals, bins=15, edgecolor='black', alpha=0.7)
axes[1].set_xlabel('残差')
axes[1].set_ylabel('频数')
axes[1].set_title('残差分布')
axes[1].grid(True, alpha=0.3)

# Q-Q 图（检查正态性）
from scipy import stats
stats.probplot(residuals, dist="norm", plot=axes[2])
axes[2].set_title('Q-Q 图')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"残差均值: {np.mean(residuals):.4f} (应接近0)")
print(f"残差标准差: {np.std(residuals):.4f}")

## 7. 实战：房价预测

In [ ]:
# 使用 sklearn 内置的糖尿病数据集（简化版）
from sklearn.datasets import load_diabetes

# 加载数据
diabetes = load_diabetes()

# 选择单个特征（BMI）
X_diabetes = diabetes.data[:, 2].reshape(-1, 1)  # BMI 指数
y_diabetes = diabetes.target  # 疾病进展指标

# 创建 DataFrame
df_diabetes = pd.DataFrame({
    'BMI': X_diabetes.flatten(),
    'disease_progression': y_diabetes
})

print("糖尿病数据集:")
print(df_diabetes.head())
print(f"\n形状: {df_diabetes.shape}")
print(f"\n统计摘要:")
print(df_diabetes.describe())

In [ ]:
# 划分数据
X_train_dia, X_test_dia, y_train_dia, y_test_dia = train_test_split(
    X_diabetes, y_diabetes, test_size=0.2, random_state=42
)

# 训练模型
model_diabetes = LinearRegression()
model_diabetes.fit(X_train_dia, y_train_dia)

# 预测
y_pred_dia = model_diabetes.predict(X_test_dia)

# 评估
r2_dia = r2_score(y_test_dia, y_pred_dia)
rmse_dia = np.sqrt(mean_squared_error(y_test_dia, y_pred_dia))

print("房价预测模型结果:")
print(f"权重: {model_diabetes.coef_[0]:.4f}")
print(f"偏置: {model_diabetes.intercept_:.4f}")
print(f"R²: {r2_dia:.4f}")
print(f"RMSE: {rmse_dia:.4f}")

In [ ]:
# 可视化
plt.figure(figsize=(10, 6))

# 训练数据
plt.scatter(X_train_dia, y_train_dia, alpha=0.5, color='blue', label='训练数据')

# 测试数据
plt.scatter(X_test_dia, y_test_dia, alpha=0.5, color='green', label='测试数据')

# 拟合线
x_line = np.linspace(X_diabetes.min(), X_diabetes.max(), 100).reshape(-1, 1)
y_line = model_diabetes.predict(x_line)
plt.plot(x_line, y_line, 'r-', linewidth=2, label='线性拟合')

plt.xlabel('BMI (标准化)')
plt.ylabel('疾病进展指标')
plt.title(f'BMI 与疾病进展的关系 (R²={r2_dia:.3f})')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 8. 总结与要点

### 核心概念

1. **线性回归模型**: $y = wx + b$
2. **最小二乘法**: 最小化预测误差的平方和
3. **正规方程**: 解析求解最优参数

### 评估指标

| 指标 | 说明 | 目标 |
|------|------|------|
| R² | 决定系数 | 接近 1 |
| MSE | 均方误差 | 越小越好 |
| RMSE | 均方根误差 | 越小越好 |
| MAE | 平均绝对误差 | 越小越好 |

### 实现方式对比

| 方式 | 优点 | 缺点 |
|------|------|------|
| 手动实现 | 理解原理 | 功能有限 |
| Scikit-learn | 功能完善、标准化 | 黑盒 |

### 注意事项

1. ⚠️ 线性回归假设特征与目标呈线性关系
2. ⚠️ 对异常值敏感
3. ⚠️ 需要进行特征缩放（多变量时）
4. ✅ 始终划分训练集和测试集
5. ✅ 检查残差分布验证模型假设

---

**下一步**: `04_multivariate_regression.ipynb` - 多元线性回归与特征工程